# Setup — add the "notifications" table to datateam_portfolio_v2

Creates the in-app notifications table (activity feed + @mentions feature) as a new table **inside the existing** datateam_portfolio_v2 feature service — it becomes the next layer/table index, alongside projects / tasks / notes / reviews / status history.

**This notebook WRITES to AGOL** (unlike the audit notebooks). Run it once. You must be the **owner or an admin** of the datateam_portfolio_v2 service.

After it runs, copy the printed table URL into the app's ARCGIS_CONFIG as notificationsUrl (src/agol.js).

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection

gis = GIS("home")
print(f"Signed in as {gis.users.me.username} @ {gis.url}")

SERVICE_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer"
flc = FeatureLayerCollection(SERVICE_URL, gis)

existing_layers = [l.properties.name for l in flc.layers]
existing_tables = [t.properties.name for t in flc.tables]
print("Layers:", existing_layers)
print("Tables:", existing_tables)

## Table definition

String fields are generously sized; is_read is an integer flag (0 = unread, 1 = read); created_at is a Date the app writes (epoch ms). An index on recipient speeds the per-user inbox query. Editor tracking is not required — actor and created_at are set by the app.

In [ ]:
TABLE_NAME = "notifications"

table_def = {
    "type": "Table",
    "name": TABLE_NAME,
    "description": "In-app notifications for the Analytics Project Tracker (activity feed + @mentions). Written by the app; read pull-based per recipient.",
    "hasAttachments": False,
    "hasM": False,
    "hasZ": False,
    "objectIdField": "OBJECTID",
    "globalIdField": "",
    "supportsAdvancedQueries": True,
    "allowGeometryUpdates": False,
    "capabilities": "Create,Delete,Query,Update,Editing",
    "fields": [
        {"name": "OBJECTID",    "type": "esriFieldTypeOID",     "alias": "OBJECTID",        "nullable": False, "editable": False},
        {"name": "notif_id",    "type": "esriFieldTypeString",  "alias": "Notification ID", "length": 50,   "nullable": True, "editable": True},
        {"name": "recipient",   "type": "esriFieldTypeString",  "alias": "Recipient",       "length": 255,  "nullable": True, "editable": True},
        {"name": "actor",       "type": "esriFieldTypeString",  "alias": "Actor",           "length": 255,  "nullable": True, "editable": True},
        {"name": "kind",        "type": "esriFieldTypeString",  "alias": "Kind",            "length": 50,   "nullable": True, "editable": True},
        {"name": "item_type",   "type": "esriFieldTypeString",  "alias": "Item Type",       "length": 20,   "nullable": True, "editable": True},
        {"name": "item_number", "type": "esriFieldTypeString",  "alias": "Item Number",     "length": 50,   "nullable": True, "editable": True},
        {"name": "item_title",  "type": "esriFieldTypeString",  "alias": "Item Title",      "length": 500,  "nullable": True, "editable": True},
        {"name": "snippet",     "type": "esriFieldTypeString",  "alias": "Snippet",         "length": 1000, "nullable": True, "editable": True},
        {"name": "is_read",     "type": "esriFieldTypeInteger", "alias": "Is Read",         "nullable": True, "editable": True, "defaultValue": 0},
        {"name": "created_at",  "type": "esriFieldTypeDate",    "alias": "Created At",       "nullable": True, "editable": True}
    ],
    "indexes": [
        {"name": "ntf_recipient_idx", "fields": "recipient", "isAscending": True, "isUnique": False, "description": "recipient lookup"}
    ]
}

print("Defined", len(table_def["fields"]), "fields for table:", TABLE_NAME)

## Create it (the write step)

Guarded so re-running is safe — if a table named notifications already exists, it does nothing.

In [ ]:
if TABLE_NAME in existing_tables:
    print(f"'{TABLE_NAME}' already exists in this service — nothing to do.")
else:
    result = flc.manager.add_to_definition({"tables": [table_def]})
    print("add_to_definition result:", result)

## Verify + get the URL

Re-reads the service and prints the new table's REST URL and fields. Copy the URL into src/agol.js as notificationsUrl.

In [ ]:
flc2 = FeatureLayerCollection(SERVICE_URL, gis)  # re-read so the new table shows
found = None
for t in flc2.tables:
    if t.properties.name == TABLE_NAME:
        found = t
        break

if not found:
    print("Could not find the new table — check the add_to_definition result above.")
else:
    print("notificationsUrl:", found.url)
    print()
    print("Fields:")
    for f in found.properties.fields:
        print(f"  {f['name']:14s} {f['type']}")
    print()
    print("Sanity query (should be 0 rows):", found.query(where="1=1", return_count_only=True))

## Next step

In src/agol.js, add to ARCGIS_CONFIG (next to projectNotesUrl):

    notificationsUrl:  '<paste the URL printed above>',

Then the app's Phase 1 notifications layer can load/write against it. To remove the table later: flc.manager.delete_from_definition({"tables": [{"name": "notifications"}]}) — or delete it from the service's Data tab in AGO.